# LME Copper Data — Cleaning and Exploratory Analysis

This notebook reads the incrementally collected Westmetall data, validates and cleans it, constructs analytical parameters, and explores relationships between copper prices and LME stock. The raw files are never modified.

## 1. Imports and adjustable parameters

In [ ]:
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

# ---------- Parameters to experiment with ----------
START_DATE = None          # Example: '2015-01-01'; None keeps all observations
END_DATE = None            # Example: '2025-12-31'; None keeps all observations
ROLLING_WINDOWS = [5, 20, 60]  # Approx. week, month, and quarter in trading days
LAG_DAYS = range(-60, 61)  # Negative: stock leads price; positive: price leads stock
MIN_PERIODS_RATIO = 0.75   # Required fraction of observations in rolling windows
WINSORIZE_RETURNS = False  # Turn on only for sensitivity analysis
WINSOR_LIMITS = (0.01, 0.99)
SAVE_PROCESSED_DATA = False  # Change to True after reviewing the transformations

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

## 2. Locate and load the raw data

In [ ]:
# This works whether Jupyter starts in the workspace, project, or notebooks directory.
candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((root / 'commodity' / 'copper' for root in candidate_roots if (root / 'commodity' / 'copper' / 'data' / 'raw' / 'lme' / 'copper_lme_raw.csv').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate data/raw/lme/copper_lme_raw.csv')

RAW_PATH = PROJECT_DIR / 'data' / 'raw' / 'lme' / 'copper_lme_raw.csv'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

raw_df = pd.read_csv(RAW_PATH, dtype='string')
print(f'Raw path: {RAW_PATH.resolve()}')
print(f'Shape: {raw_df.shape}')
raw_df.head()

In [ ]:
raw_df.info()
display(raw_df.tail())
display(raw_df.isna().sum().rename('empty_cells'))
display(raw_df.nunique().rename('unique_values'))

## 3. Clean types and validate

A dash in the source means unavailable, so it becomes `NaN`. Numeric strings retain their original form in `raw_df`; cleaning happens only in `df`.

In [ ]:
df = raw_df.copy()
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='raise')

numeric_columns = ['cash_settlement', 'three_month', 'stock']
for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column].replace('-', pd.NA).str.replace(',', '', regex=False),
        errors='raise',
    )

df['source_year'] = pd.to_numeric(df['source_year'], errors='raise').astype('int16')
df['fetched_at_utc'] = pd.to_datetime(df['fetched_at_utc'], utc=True, errors='raise')
df = df.sort_values('date').reset_index(drop=True)

if START_DATE is not None:
    df = df.loc[df['date'] >= pd.Timestamp(START_DATE)].copy()
if END_DATE is not None:
    df = df.loc[df['date'] <= pd.Timestamp(END_DATE)].copy()

df.head()

In [ ]:
quality_report = pd.Series({
    'rows': len(df),
    'first_date': df['date'].min(),
    'last_date': df['date'].max(),
    'duplicate_dates': df['date'].duplicated().sum(),
    'weekend_dates': (df['date'].dt.dayofweek >= 5).sum(),
    'missing_cash': df['cash_settlement'].isna().sum(),
    'missing_three_month': df['three_month'].isna().sum(),
    'missing_stock': df['stock'].isna().sum(),
    'source_year_mismatches': (df['date'].dt.year != df['source_year']).sum(),
    'nonpositive_numeric_values': (df[numeric_columns] <= 0).sum().sum(),
})
display(quality_report.to_frame('value'))

assert df['date'].is_monotonic_increasing
assert not df['date'].duplicated().any()
assert (df['date'].dt.year == df['source_year']).all()
assert not (df[numeric_columns] <= 0).any().any()

In [ ]:
# Review exceptional observations instead of silently removing them.
exception_rows = df.loc[
    df[numeric_columns].isna().any(axis=1) | (df['date'].dt.dayofweek >= 5),
    ['date', *numeric_columns, 'source_url'],
]
exception_rows

## 4. Construct analytical parameters

Returns and changes are preferable to raw levels for many correlation tests because price and inventory levels can both trend over time. The spread is cash minus three-month price; a positive value indicates backwardation and a negative value indicates contango.

In [ ]:
df['spread'] = df['cash_settlement'] - df['three_month']
df['spread_pct'] = 100 * df['spread'] / df['three_month']
df['cash_return'] = df['cash_settlement'].pct_change(fill_method=None)
df['cash_log_return'] = np.log(df['cash_settlement']).diff()
df['three_month_return'] = df['three_month'].pct_change(fill_method=None)
df['stock_change'] = df['stock'].diff()
df['stock_pct_change'] = df['stock'].pct_change(fill_method=None)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()

for window in ROLLING_WINDOWS:
    minimum = max(2, int(window * MIN_PERIODS_RATIO))
    df[f'cash_ma_{window}d'] = df['cash_settlement'].rolling(window, min_periods=minimum).mean()
    df[f'cash_volatility_{window}d'] = df['cash_log_return'].rolling(window, min_periods=minimum).std() * np.sqrt(252)
    df[f'stock_ma_{window}d'] = df['stock'].rolling(window, min_periods=minimum).mean()

if WINSORIZE_RETURNS:
    for column in ['cash_return', 'cash_log_return', 'three_month_return', 'stock_pct_change']:
        lower, upper = df[column].quantile(WINSOR_LIMITS)
        df[column] = df[column].clip(lower, upper)

df[['date', 'cash_settlement', 'three_month', 'stock', 'spread', 'cash_return', 'stock_pct_change']].tail()

## 5. Distributions and descriptive statistics

In [ ]:
analysis_columns = [
    'cash_settlement', 'three_month', 'stock', 'spread', 'spread_pct',
    'cash_return', 'cash_log_return', 'stock_change', 'stock_pct_change',
]
display(df[analysis_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=(
    'Cash-settlement price', 'Daily log return', 'LME stock', 'Cash minus three-month spread'
))
for row, col, series, color in [
    (1, 1, df['cash_settlement'], '#1976D2'),
    (1, 2, df['cash_log_return'], '#C62828'),
    (2, 1, df['stock'], '#00897B'),
    (2, 2, df['spread'], '#EF6C00'),
]:
    fig.add_trace(go.Histogram(x=series.dropna(), marker_color=color, showlegend=False), row=row, col=col)
fig.update_layout(title='LME copper distributions', height=750, template='plotly_white', bargap=0.04)
fig.show()

## 6. Time-series behavior

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    subplot_titles=('LME copper prices', 'LME copper stock', 'Cash minus three-month spread'))
for column, name, color in [('cash_settlement', 'Cash settlement', '#1976D2'),
                            ('three_month', 'Three month', '#EF6C00')]:
    fig.add_trace(go.Scatter(x=df['date'], y=df[column], name=name, line_color=color), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['stock'], name='Stock', line_color='#00897B'), row=2, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['spread'], name='Spread', line_color='#7B1FA2'), row=3, col=1)
fig.add_hline(y=0, line_color='#455A64', row=3, col=1)
fig.update_yaxes(title_text='USD/tonne', row=1, col=1)
fig.update_yaxes(title_text='Tonnes', row=2, col=1)
fig.update_yaxes(title_text='USD/tonne', row=3, col=1)
fig.update_layout(height=900, template='plotly_white', hovermode='x unified')
fig.show()

## 7. Correlations

Pearson measures linear association; Spearman measures monotonic association and is less sensitive to extreme observations. Correlation does not establish causality.

In [ ]:
level_columns = ['cash_settlement', 'three_month', 'stock', 'spread']
change_columns = ['cash_return', 'three_month_return', 'stock_pct_change', 'spread_pct']
pearson_levels = df[level_columns].corr(method='pearson')
spearman_levels = df[level_columns].corr(method='spearman')
pearson_changes = df[change_columns].corr(method='pearson')
spearman_changes = df[change_columns].corr(method='spearman')
fig = make_subplots(rows=2, cols=2, subplot_titles=(
    'Pearson: levels', 'Spearman: levels', 'Pearson: changes', 'Spearman: changes'))
for (row, col), matrix in zip([(1, 1), (1, 2), (2, 1), (2, 2)],
    [pearson_levels, spearman_levels, pearson_changes, spearman_changes]):
    fig.add_trace(go.Heatmap(z=matrix.to_numpy(), x=matrix.columns, y=matrix.index,
        zmin=-1, zmax=1, colorscale='RdBu', reversescale=True,
        text=matrix.round(2).astype(str).to_numpy(), texttemplate='%{text}',
        showscale=(row, col) == (1, 2)), row=row, col=col)
fig.update_layout(title='Copper price and stock correlations', height=900, template='plotly_white')
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Price level vs stock level', 'Daily price return vs stock change'))
for col, x, y, color in [(1, 'stock', 'cash_settlement', '#1976D2'),
                          (2, 'stock_pct_change', 'cash_return', '#00897B')]:
    sample = df[[x, y]].dropna()
    fig.add_trace(go.Scattergl(x=sample[x], y=sample[y], mode='markers',
        marker=dict(size=5, color=color, opacity=0.25), showlegend=False), row=1, col=col)
    slope, intercept = np.polyfit(sample[x].to_numpy(dtype=float), sample[y].to_numpy(dtype=float), 1)
    endpoints = [sample[x].min(), sample[x].max()]
    fig.add_trace(go.Scatter(x=endpoints, y=[slope * value + intercept for value in endpoints],
        mode='lines', line_color='#C62828', name='Linear fit', showlegend=col == 1), row=1, col=col)
fig.update_layout(title='Copper price and stock relationships', height=500, template='plotly_white')
fig.show()

## 8. Lagged relationships

This checks whether changes in stock tend to precede or follow price returns. It uses observation lags (trading rows), not exact calendar days.

In [ ]:
lag_correlations = pd.DataFrame({
    'lag': list(LAG_DAYS),
    'pearson': [df['cash_return'].corr(df['stock_pct_change'].shift(lag), method='pearson') for lag in LAG_DAYS],
    'spearman': [df['cash_return'].corr(df['stock_pct_change'].shift(lag), method='spearman') for lag in LAG_DAYS],
})
display(lag_correlations.loc[lag_correlations['pearson'].abs().nlargest(10).index].sort_values('lag'))
fig = go.Figure()
for column, color in [('pearson', '#1976D2'), ('spearman', '#00897B')]:
    fig.add_trace(go.Scatter(x=lag_correlations['lag'], y=lag_correlations[column],
        name=column.title(), line_color=color))
fig.add_hline(y=0, line_color='#455A64')
fig.add_vline(x=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Lag correlation: price return vs shifted stock change',
    xaxis_title='Stock-change lag (trading observations)', yaxis_title='Correlation',
    height=500, template='plotly_white')
fig.show()

## 9. Correlation stability over time

In [ ]:
yearly_summary = (
    df.groupby('year')
      .agg(
          observations=('date', 'size'),
          average_cash=('cash_settlement', 'mean'),
          year_end_cash=('cash_settlement', 'last'),
          average_stock=('stock', 'mean'),
          average_spread=('spread', 'mean'),
          annualized_volatility=('cash_log_return', lambda x: x.std() * np.sqrt(252)),
      )
)
yearly_summary['price_stock_level_corr'] = df.groupby('year')[['cash_settlement', 'stock']].apply(
    lambda group: group['cash_settlement'].corr(group['stock'])
)
yearly_summary['return_stock_change_corr'] = df.groupby('year')[['cash_return', 'stock_pct_change']].apply(
    lambda group: group['cash_return'].corr(group['stock_pct_change'])
)
display(yearly_summary)

In [ ]:
monthly = (
    df.set_index('date')
      .resample('ME')
      .agg({
          'cash_settlement': 'last',
          'three_month': 'last',
          'stock': 'last',
          'spread': 'mean',
          'cash_log_return': 'std',
      })
)
monthly['cash_monthly_return'] = monthly['cash_settlement'].pct_change(fill_method=None)
monthly['stock_monthly_change'] = monthly['stock'].pct_change(fill_method=None)
monthly['monthly_volatility'] = monthly['cash_log_return'] * np.sqrt(252)

display(monthly.tail(12))
display(monthly[['cash_monthly_return', 'stock_monthly_change', 'spread']].corr())

## 10. Optional processed-data export

Review all transformations first. When satisfied, set `SAVE_PROCESSED_DATA = True` in the parameter cell and rerun.

In [ ]:
if SAVE_PROCESSED_DATA:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    processed_path = PROCESSED_DIR / 'copper_lme_processed.csv'
    df.to_csv(processed_path, index=False)
    print(f'Saved {len(df):,} rows to {processed_path.resolve()}')
else:
    print('Export skipped. Set SAVE_PROCESSED_DATA = True when ready.')

## Standard market dashboard

This governed, read-only section uses the same presentation contract across commodity projects:
source coverage, physical and certificate activity, separate price panels, physical-goods
composition, and validated processed bubbles. It never writes raw data or constructs a missing
bubble. For Copper, product comparability still follows the project-specific workflow.

In [ ]:
from pathlib import Path
import sys

def locate_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "commodity" / "copper").exists() and (candidate / "shared").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

WORKSPACE_ROOT = locate_workspace()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from shared.notebook_tools.commodity_dashboard import (
    goods_type_counts,
    load_markets,
    market_summary,
    plot_available_bubbles,
    plot_goods_type_counts,
    plot_market_prices,
    plot_trade_activity,
)

PROJECT_DIR = WORKSPACE_ROOT / "commodity" / "copper"
physical_dashboard, certificate_dashboard = load_markets(
    PROJECT_DIR, "copper", physical_filename='copper_cathode_physical_raw.csv'
)
display(market_summary(physical_dashboard, certificate_dashboard))
plot_trade_activity(physical_dashboard, certificate_dashboard, "Copper")
plot_market_prices(physical_dashboard, certificate_dashboard, "Copper")
goods_count_table = plot_goods_type_counts(physical_dashboard, "Copper", top_n=30)
display(goods_count_table)
bubble_series_plotted = plot_available_bubbles(PROJECT_DIR, "Copper")

## Historical bubble distribution

This section reads the standardized processed table and renders an interactive Plotly figure for
each bubble type. The panels show the observed distribution, empirical cumulative distribution
function F(x), and magnitude frequency P(|Bubble| >= |x|). Negative bubbles retain their sign in
the first two panels; the third panel measures magnitude only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def locate_distribution_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "shared").exists() and (candidate / "commodity/copper").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

distribution_workspace = locate_distribution_workspace()
if str(distribution_workspace) not in sys.path:
    sys.path.insert(0, str(distribution_workspace))

from shared.market_analysis.bubble_distribution import plot_distribution_plotly

distribution_project = distribution_workspace / "commodity/copper"
distribution_files = list(
    (distribution_project / "data/processed/bubble").glob("*_bubble_distribution.csv")
)
if len(distribution_files) != 1:
    raise ValueError(f"Expected one named bubble distribution CSV, found {distribution_files}")
bubble_distribution = pd.read_csv(distribution_files[0], parse_dates=["observation_date"])
for series_id, series_distribution in bubble_distribution.groupby("series_id", sort=True):
    comparison = series_distribution["comparison"].iloc[0]
    figure = plot_distribution_plotly(series_distribution, comparison)
    figure.show()

display(bubble_distribution)